# Notebook 06 · La capa `Embedding` de Keras

> **Unidad 2 · NLP e IA Generativa — Clase 1: Text Mining y Embeddings**  
> Ruta de la clase: 01 Texto → 02 Limpieza → 03 Tokenización → 04 Padding → 05 BoW/TF-IDF → **06 Embeddings** → 07 Clasificador → 08 Laboratorio

## Objetivo
Entender la capa `Embedding` como una **tabla de búsqueda entrenable** $W \in \mathbb{R}^{V \times d}$, interpretar sus parámetros y *shapes*, y **ver** cómo se organizan los vectores después de entrenar.

- $V$ = `input_dim` (tamaño del vocabulario + 1 por el padding)
- $d$ = `output_dim` (dimensión de cada vector)

```
índices (batch, maxlen)  ──►  Embedding  ──►  vectores (batch, maxlen, d)
```

⏱️ Tiempo sugerido: 20 min

> 🧑‍🎓 **Versión para estudiantes.** Las celdas de código están vacías: complétalas en clase siguiendo las explicaciones.
> Cada celda trae **pistas** (funciones útiles) y los **nombres de variables** que deben quedar definidos, porque las celdas siguientes los usan.
> Ejecuta las celdas en orden. La versión resuelta está en la carpeta `notebooks/`.

In [47]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense


# Dataset ampliado para demostración
textos = [
    # --- Ejemplos Positivos (1) ---
    "excelente clase de nlp",
    "contenido muy claro y util",
    "laboratorio excelente y practico",
    "muy buena introduccion a embeddings",
    "el profesor explica con gran claridad y paciencia",
    "los ejemplos practicos facilitaron el aprendizaje",
    "gran dominio del tema y material impecable",
    "la dinamica del curso supero mis expectativas",
    "muy recomendado el taller para principiantes",
    "los ejercicios de programacion fueron enriquecedores",
    "resolvio todas las dudas con ejemplos reales",
    "material didactico completo y bien organizado",
    
    # --- Ejemplos Negativos (0) ---
    "explicacion confusa y lenta",
    "no me gusto la clase",
    "el ritmo fue demasiado rapido y poco claro",
    "falto profundidad en los conceptos teoricos",
    "presentacion aburrida y con muchos errores",
    "los laboratorios no funcionaban correctamente",
    "explicacion superficial y poco dominio del tema",
    "no se entendio el objetivo de la sesion",
    "mala conexion y audio deficiente durante la sesion",
    "ejemplos desactualizados y poco aplicables",
    "tiempo insuficiente para resolver inquietudes",
    "el contenido fue tedioso y mal estructurado",
]

# Etiquetas correspondientes (1: Positivo, 0: Negativo)
etiquetas = np.array([
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,  # 12 positivos
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0   # 12 negativos
])

## 1) Una Embedding es una tabla de búsqueda
Antes de entrenar, los vectores son **aleatorios**. Pedir el embedding del índice 3 es literalmente leer la **fila 3** de la matriz de pesos.

In [48]:
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
tokenizer.fit_on_texts(textos)
secuencias = tokenizer.texts_to_sequences(textos)

input_length = 6
x = pad_sequences(secuencias, maxlen=input_length, padding="post", truncating="post")

print("Shape de entrada (x):", x.shape)
print("Ejemplo de secuencia padded:", x[0])

Shape de entrada (x): (24, 6)
Ejemplo de secuencia padded: [13 14  6 25  0  0]


## 2) Datos: opiniones de clientes
Mismo pipeline de los notebooks anteriores: limpieza → tokenización → padding.

In [49]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Input

vocab_size = len(tokenizer.word_index) + 1
output_dim = 64
input_len = 6

model = Sequential([
    Input(shape=(input_len,)),
    Embedding(input_dim=vocab_size, output_dim=output_dim, input_length=input_len),
    GlobalAveragePooling1D(),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 6, 64)          │         6,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_10     │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 8)              │           520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,865 (26.82 KB)

 Trainable params: 6,865 (26.82 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(x, etiquetas, epochs=10, verbose=0)
print("Modelo entrenado.")
print(rouded_history := {k: [round(v, 4) for v in vals] for k, vals in history.history.items()})

embedding_layer = model.layers[0]
muestra = x[:1]
salida = embedding_layer(muestra)

Modelo entrenado.
{'accuracy': [0.5833, 0.7083, 0.875, 0.9583, 0.9583, 1.0, 1.0, 1.0, 1.0, 1.0], 'loss': [0.691, 0.6874, 0.684, 0.6806, 0.6773, 0.6739, 0.6706, 0.6673, 0.6638, 0.6602]}


## 4) Shapes paso a paso

In [60]:
nuevo_texto = "No me gusto la clase, fue confusa y aburrida"

seq = tokenizer.texts_to_sequences([nuevo_texto])
padded_seq = pad_sequences(seq, maxlen=input_len, padding="post", truncating="post")

prob = model.predict(padded_seq)[0][0]
print(f"Texto", nuevo_texto, "probabilidad de ser positivo:", round(prob, 4))
print("Predicción:", "Positivo" if prob >= 0.5 else "Negativo")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
Texto No me gusto la clase, fue confusa y aburrida probabilidad de ser positivo: 0.4743
Predicción: Negativo


## 5) Entrenar y comparar vectores antes / después
Guardamos los pesos antes de entrenar para ver cómo **el entrenamiento organiza el espacio**.

In [52]:
# ✍️ Ejercicio 5: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): copy, fit, get_weights
# 📌 Al terminar deberían existir: W_antes, history, W_despues


In [53]:
# ✍️ Ejercicio 6: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): DataFrame, append, cosine_similarity
# 📌 Al terminar deberían existir: vec(), pares, filas


> 🔎 Después de entrenar, palabras con el **mismo sentimiento** tienden a coseno positivo y las opuestas a coseno negativo. El modelo aprendió una "dirección de sentimiento" en el espacio, **solo a partir de las etiquetas**.

## 6) Visualizar el espacio de embeddings (PCA a 2D)

In [54]:
# ✍️ Ejercicio 7: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): subplots, suptitle, tight_layout, show, fit_transform, scatter, set_title, grid
# 📌 Al terminar deberían existir: palabras, fig, axes


## 7) Predicción sobre texto nuevo
Siempre el **mismo** pipeline: `limpiar_texto` → `tokenizer` → `pad_sequences`.

In [55]:
# ✍️ Ejercicio 8: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): pad_sequences, texts_to_sequences, limpiar_texto, predict


## Para llevar
- La Embedding **no sabe nada** al inicio: aprende de la tarea (aquí, sentimiento).
- Con más datos y tareas más generales (Word2Vec, GloVe, FastText) los vectores capturan relaciones semánticas ricas.
- Limitación: **un vector por palabra**, sin importar el contexto (`banco` de dinero vs. `banco` del parque) → lo resuelven los **Transformers** (próxima clase).

## Preguntas de reflexión
1. ¿Qué efecto tendría duplicar `output_dim`? ¿Y en el número de parámetros?
2. ¿Qué sucede si `input_dim` es menor que el índice más grande del vocabulario?
3. ¿Por qué la salida de Embedding tiene 3 dimensiones?
4. ¿Qué hace `mask_zero=True` y por qué importa con `GlobalAveragePooling1D`?